In [31]:
import hopsworks
import os
import dotenv

dotenv.load_dotenv()

# Load Feature Group
feature_group_name = "weather_prediction"
features = ['temperature_2m', 'relative_humidity_2m', 'precipitation',
       'wind_speed_10m', 'cloud_cover', 'surface_pressure', 'dew_point_2m', 'pressure_msl',
       'derived_hour', 'derived_month', 'relative_humidity_2m_24h_avg', 'pressure_msl_3h_pct_change']
labels = ["precipation_label"]
features_and_labels = features.extend(labels)

project = hopsworks.login(
    project='mlops',  # Replace with your project name
    host="eu-west.cloud.hopsworks.ai",
    port=443,
    api_key_value=os.environ["HOPSWORKS_API_KEY"]  # Get from Hopsworks UI > Account Settings > API Keys
)

fs = project.get_feature_store()
# Get all versions of the feature group
fgs = fs.get_feature_groups()
fg_versions = [fg.version for fg in fgs if fg.name == feature_group_name]

# Load latest version of feature group as DataFrame
fg = fs.get_feature_group(name=feature_group_name, version=max(fg_versions))
df = fg.read()

print(f"Loaded {feature_group_name} with version {max(fg_versions)}")
print(f"Features and Labels:\n{fg.features}")
print(df.columns)

2026-04-06 21:24:21,191 INFO: Closing external client and cleaning up certificates.
2026-04-06 21:24:21,192 INFO: Connection closed.
2026-04-06 21:24:21,193 INFO: Initializing external client
2026-04-06 21:24:21,194 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-04-06 21:24:22,506 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/31926
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.27s) 
Loaded weather_prediction with version 2
Features and Labels:
[Feature('partition', 'string', None, False, False, True, 'varchar(100)', None, 37866), Feature('temperature_2m', 'float', None, False, False, False, 'float', None, 37866), Feature('relative_humidity_2m', 'float', None, False, False, False, 'float', None, 37866), Feature('precipitation', 'float', None, False, False, False, 'float', None, 37866), Feature('precipitation_probability', 'float', None, False, False, False, 'float', None, 37

In [37]:
# Create Feature View with features and labels

query = fg.select(features)

feature_view = fs.create_feature_view(
    name="weather_prediction",
    version=fg.version, # Always use the same version number als the referenced feature group
    query=query,
    labels=labels,
    description="Feature View for weather prediction"
)


RestAPIError: Metadata operation error: (url: https://eu-west.cloud.hopsworks.ai/hopsworks-api/api/project/31926/featurestores/20610/featureview). Server response: 
HTTP code: 400, HTTP reason: Bad Request, body: b'{"errorCode":270179,"usrMsg":"Feature view: weather_prediction, version: 2","errorMsg":"The provided feature view name and version already exists"}', error code: 270179, error msg: The provided feature view name and version already exists, user msg: Feature view: weather_prediction, version: 2

In [ ]:
# Create the training and test datasets
# Random DataSets need to be created outside of Hopsworks
from sklearn.model_selection import train_test_split
import pandas as pd


# # Separate features and labels
# X = pd.DataFrame
# y = pd.DataFrame

# for label in labels:
#     X = X + df.drop(label, axis=1)  # Features
#     y = y + df[label]             # Labels

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,  # For reproducibility
    stratify=y         # Optional: ensures balanced class distribution
)

TypeError: Cannot broadcast np.ndarray with operand of type <class 'type'>